In [ ]:
import json
import os
import time

import cmasher as cmr
import gc_utils
import gizmo_analysis as gizmo
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import utilities as ut
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogLocator
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

# SMALL_SIZE = 10
# MEDIUM_SIZE = 10
# BIGGER_SIZE = 10

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


# Preamble

In [ ]:
# sim = "m12i"
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

sim_dir = "/Users/z5114326/Documents/simulations/"
sim_dir = "/Volumes/Expansion/simulations/"
sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

In [ ]:
all_data_fil = sim_dir + "/" + "m12i" + "/" + "m12i" + "_res7100/" + "snapshot_times.txt"
all_data = pd.read_table(all_data_fil, comment="#", header=None, sep=r"\s+")
all_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
all_times = np.array(all_data["time_Gyr"])
all_snaps = np.array(all_data["index"])

public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
timez0 = float(pub_data[pub_data["index"] == 600]["time_Gyr"].values[0])
pub_times = np.array(pub_data["time_Gyr"])
pub_snaps = np.array(pub_data["index"])

# Functions

In [ ]:
def get_halo_center(part, halt, sim, sim_dir, snapshot):
    # check if centering should be on dm halo
    sim_codes = sim_dir + "simulation_codes.json"
    with open(sim_codes) as sim_json:
        sim_data = json.load(sim_json)
    main_halo_tid = sim_data[sim]["halo"]

    not_host_snap_lst = gc_utils.get_different_snap_lst(main_halo_tid, halt, sim, sim_dir)

    # is the MW progenitor is the main host at this snapshot
    is_main_host = snapshot not in not_host_snap_lst

    halt_center_snap_lst = sim_data[sim]["dm_center"]
    use_dm_center = snapshot in halt_center_snap_lst

    # if the halo is not the host at this snapshot or it has been flagged to use dm center at this snapshot
    if (not is_main_host) or (use_dm_center):
        # get MW progenitor halo details at this snapshot
        halo_tid = gc_utils.get_main_prog_at_snap(halt, main_halo_tid, snapshot)

        if use_dm_center:
            halo_detail_dict = gc_utils.get_dm_halo_details(part, halt, halo_tid, snapshot, True)
        else:
            halo_detail_dict = gc_utils.get_halo_details(part, halt, halo_tid, snapshot)

        return_dict = {"use_host_prop": False, "halo_details": halo_detail_dict}

    else:
        return_dict = {"use_host_prop": True}

    return return_dict


def get_densities_and_ratios(part, halt, sim, sim_dir, snapshot, rlim=3, temp_lim=4):
    host_return_dict = get_halo_center(part, halt, sim, sim_dir, snapshot)

    # stellar_mass
    if host_return_dict["use_host_prop"]:
        star_msk = part["star"].prop("host.distance.principal.total") < rlim
        dark_msk = part["dark"].prop("host.distance.principal.total") < rlim
        gas_msk = part["gas"].prop("host.distance.principal.total") < rlim
    else:
        halo_detail_dict = host_return_dict["halo_details"]

        star_msk = (
            ut.particle.get_distances_wrt_center(
                part,
                species=["star"],
                center_position=halo_detail_dict["position"],
                rotation=halo_detail_dict["rotation"],
                coordinate_system="cartesian",
                total_distance=True,
            )
            < rlim
        )

        dark_msk = (
            ut.particle.get_distances_wrt_center(
                part,
                species=["dark"],
                center_position=halo_detail_dict["position"],
                rotation=halo_detail_dict["rotation"],
                coordinate_system="cartesian",
                total_distance=True,
            )
            < rlim
        )

        gas_msk = (
            ut.particle.get_distances_wrt_center(
                part,
                species=["gas"],
                center_position=halo_detail_dict["position"],
                rotation=halo_detail_dict["rotation"],
                coordinate_system="cartesian",
                total_distance=True,
            )
            < rlim
        )

    gas_tmsk = np.log10(part["gas"]["temperature"]) < temp_lim

    volume = (4 / 3) * np.pi * rlim**3

    star_mass = np.sum(part["star"]["mass"][star_msk])
    dark_mass = np.sum(part["dark"]["mass"][dark_msk])
    gas_mass = np.sum(part["gas"]["mass"][gas_msk])
    cold_gas_mass = np.sum(part["gas"]["mass"][gas_msk & gas_tmsk])

    mass_rat = star_mass / dark_mass
    bary_rat = (star_mass + gas_mass) / dark_mass

    star_den = star_mass / volume
    gas_den = gas_mass / volume
    cold_gas_den = cold_gas_mass / volume

    return mass_rat, star_den, bary_rat, gas_den, cold_gas_den


def get_sfr(part, tz0_gyr, dt_i_myr, dt_a_myr, rlim):
    print("Getting sfr")
    dt_i_yr = dt_i_myr * 1e6
    dt_a_yr = dt_a_myr * 1e6

    rmsk = part["star"].prop("form.host.distance.total") < rlim
    tfor = part["star"].prop("form.time", rmsk) * 1e9  # express in yr
    mfor = part["star"].prop("form.mass", rmsk)

    t_grid_i_yr = np.arange(0, tz0_gyr * 1e9, dt_i_yr)
    t_grid_a_yr = np.arange(0, tz0_gyr * 1e9, dt_a_yr)

    sfr_i = np.zeros_like(t_grid_i_yr)
    sfr_a = np.zeros_like(t_grid_a_yr)

    for i, t in enumerate(t_grid_i_yr):
        msk = (tfor >= t - dt_i_yr / 2) & (tfor < t + dt_i_yr / 2)
        sfr_i[i] = mfor[msk].sum() / dt_i_yr

    for i, t in enumerate(t_grid_a_yr):
        msk = (tfor >= t - dt_a_yr / 2) & (tfor < t + dt_a_yr / 2)
        sfr_a[i] = mfor[msk].sum() / dt_a_yr

    lbt_i = tz0_gyr - t_grid_i_yr * 1e-9
    lbt_a = tz0_gyr - t_grid_a_yr * 1e-9

    return lbt_i, sfr_i, lbt_a, sfr_a


def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert(v) for v in obj]
    return obj


# Workings

In [ ]:
rlim_d = 3  # kpc
rlim_s = 20  # kpc

dt_i_myr = 10  # Myr
dt_a_myr = 250  # Myr

p3 = 1.0  # threshold

atims = all_times[all_snaps >= 20]
asnps = all_snaps[all_snaps >= 20]

ptims = np.append(pub_times[(pub_snaps > 45) & (pub_snaps < 590)], pub_times[-1])
psnps = np.append(pub_snaps[(pub_snaps > 45) & (pub_snaps < 590)], pub_snaps[-1])

gal_dict = {
    sim: {
        "stellar_mass": {"lbt": None, "dat": None},
        "dark_mass": {"lbt": None, "dat": None},
        "stellar_density": {"lbt": None, "dat": None},
        "sfr": {"ins": {"lbt": None, "dat": None}, "avg": {"lbt": None, "dat": None}},
        "mass_ratio": {"lbt": None, "dat": None},
        "specific_accretion": {"lbt": None, "dat": None},
        "bary_ratio": {"lbt": None, "dat": None},
        "gas_density": {"lbt": None, "dat": None},
        "cold_gas_density": {"lbt": None, "dat": None},
    }
    for sim in sim_lst
}


In [ ]:
gal_path = "../data/sim_data/galaxy_dict.json"
if os.path.exists(gal_path):
    print("Loading existing gal_dict JSON...")
    with open(gal_path, "r", encoding="utf-8") as f:
        gal_dict = json.load(f)

else:
    for sim in sim_lst:
        fir_dir = sim_dir + sim + "/" + sim + "_res7100"

        halt = gc_utils.get_halo_tree(sim, sim_dir, species=["star"])
        host_tid = sim_data[sim]["halo"]

        sm_lst = []  # stellar mass list
        dm_lst = []  # dark mass list (for specific accretion)
        sd_lst = []  # stellar density list
        mr_lst = []  # mass ratio list
        br_lst = []  # batyonic mass ratio list
        gd_lst = []  # gas density list
        cgd_lst = []  # cold (< 10^4) gas density list

        for asnp in asnps:
            tid = gc_utils.get_halo_prog_at_snap(halt, host_tid, asnp)
            idx = np.where(halt["tid"] == tid)[0][0]
            sm_lst.append(halt["star.mass"][idx])
            dm_lst.append(halt["mass"][idx])

        for psnp in psnps:
            start_clock = time.perf_counter()
            print(sim, "-", psnp)
            part = gizmo.io.Read.read_snapshots(
                ["star", "dark", "gas"],
                "index",
                psnp,
                fir_dir,
                assign_hosts_rotation=True,
            )

            mass_rat, star_den, bary_rat, gas_den, cold_gas_den = get_densities_and_ratios(
                part, halt, sim, sim_dir, psnp, rlim=rlim_d
            )
            sd_lst.append(star_den)
            mr_lst.append(mass_rat)
            br_lst.append(bary_rat)
            gd_lst.append(gas_den)
            cgd_lst.append(cold_gas_den)
            end_clock = time.perf_counter()
            print(f"Snapshot time: {end_clock - start_clock} seconds")

        # need to treat snapshot 600 seperately for this
        part = gizmo.io.Read.read_snapshots(
            ["star"],
            "index",
            600,
            fir_dir,
            assign_hosts_rotation=True,
            assign_formation_coordinates=True,
        )
        lbt_i, sfr_i, lbt_a, sfr_a = get_sfr(part, timez0, dt_i_myr, dt_a_myr, rlim_s)

        sm_arr = np.array(sm_lst)
        dm_arr = np.array(dm_lst)

        dm_grad_arr = np.gradient(dm_arr[1:], atims[1:])
        sa_arr = dm_grad_arr / dm_arr[1:]  # specific accretion

        sd_arr = np.array(sd_lst)
        mr_arr = np.array(mr_lst)

        gd_arr = np.array(gd_lst)
        br_arr = np.array(br_lst)

        cgd_arr = np.array(cgd_lst)

        gal_dict[sim]["stellar_mass"]["dat"] = sm_arr
        gal_dict[sim]["stellar_mass"]["lbt"] = timez0 - atims

        gal_dict[sim]["dark_mass"]["dat"] = dm_arr
        gal_dict[sim]["dark_mass"]["lbt"] = timez0 - atims

        gal_dict[sim]["specific_accretion"]["dat"] = sa_arr
        gal_dict[sim]["specific_accretion"]["lbt"] = timez0 - atims[1:]

        gal_dict[sim]["stellar_density"]["dat"] = sd_arr
        gal_dict[sim]["stellar_density"]["lbt"] = timez0 - ptims

        gal_dict[sim]["mass_ratio"]["dat"] = mr_arr
        gal_dict[sim]["mass_ratio"]["lbt"] = timez0 - ptims

        gal_dict[sim]["bary_ratio"]["dat"] = br_arr
        gal_dict[sim]["bary_ratio"]["lbt"] = timez0 - ptims

        gal_dict[sim]["gas_density"]["dat"] = gd_arr
        gal_dict[sim]["gas_density"]["lbt"] = timez0 - ptims

        gal_dict[sim]["cold_gas_density"]["dat"] = cgd_arr
        gal_dict[sim]["cold_gas_density"]["lbt"] = timez0 - ptims

        gal_dict[sim]["sfr"]["ins"]["dat"] = sfr_i
        gal_dict[sim]["sfr"]["ins"]["lbt"] = lbt_i

        gal_dict[sim]["sfr"]["avg"]["dat"] = sfr_a
        gal_dict[sim]["sfr"]["avg"]["lbt"] = lbt_a

    gal_dict_json = convert(gal_dict)

    with open(gal_path, "w") as f:
        json.dump(gal_dict_json, f, indent=4)

# Plotting

In [ ]:
to_path = "../data/sim_data/turn_offs_lbt.json"
with open(to_path, "r", encoding="utf-8") as f:
    to_dict = json.load(f)

su_path = "../data/sim_data/spin_up_lbt.json"
with open(su_path, "r", encoding="utf-8") as f:
    su_dict = json.load(f)

bf_path = "../data/sim_data/boundary_formation_lbt.json"
with open(bf_path, "r", encoding="utf-8") as f:
    bf_dict = json.load(f)

bo_path = "../data/sim_data/boundary_onset_lbt.json"
with open(bo_path, "r", encoding="utf-8") as f:
    bo_dict = json.load(f)

In [ ]:
# fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
# plt.subplots_adjust(hspace=0.05, wspace=0.05)

fig, axs = plt.subplots(3, 2, figsize=(3.5, 3.75))
plt.subplots_adjust(hspace=0.075, wspace=0.075)

for i, ax in enumerate(axs.flatten()):
    if i > 4:
        ax.axis("off")
        continue

    sim = sim_lst[i]

    l1_txt = r"10 Myr Average"
    l2_txt = r"250 Myr Average"
    l3_txt = r"N$_{\mathrm{GC}}$ - M$_{\mathrm{halo}}$ Turn-Off Time"
    l4_txt = r"Survival Boundary Formation Time"

    (l1,) = ax.plot(
        gal_dict[sim]["sfr"]["ins"]["lbt"],
        gal_dict[sim]["sfr"]["ins"]["dat"],
        c="lightblue",
        label=l1_txt,
        lw=0.75,
    )
    (l2,) = ax.plot(
        gal_dict[sim]["sfr"]["avg"]["lbt"],
        gal_dict[sim]["sfr"]["avg"]["dat"],
        c="blue",
        ls="--",
        label=l2_txt,
        lw=1,
    )
    l3 = ax.axvline(to_dict["cnt"][sim], c="g", ls="--", lw=1, label=l3_txt)
    l4 = ax.axvline(bf_dict["in_situ"][sim], c="r", ls="dotted", lw=1, label=l4_txt)
    ax.axvline(bo_dict["in_situ"][sim], c="r", ls="dotted", lw=1)

    # ax.axhline(10, xmax=bo_dict["in_situ"][sim], c="r", ls="dotted", lw=2)

    # ax.set_xlim([0, time])
    ax.set_xlim([timez0, 0])
    ax.set_ylim([7 * 1e-2, 7 * 1e1])

    ax.set_xticks([0, 2, 4, 6, 8, 10, 12])
    ax.set_yscale("log")

    if i < 3:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Lookback Time (Gyr)")

    if i not in [0, 2, 4]:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel(r"SFR (M$_{\odot}$ yr$^{-1}$)")

    ax.text(0.77, 0.15, sim, ha="left", va="center", transform=ax.transAxes)

# Put legend in empty subplot
legend_ax = axs.flatten()[5]
legend_ax.axis("off")
# legend_ax.legend(handles=[l1, l2, l3, l4], loc="center", frameon=False)
# legend_ax.legend(handles=[l1, l2, l4], loc="center", frameon=False)

sfr_name = "../data/figures/galaxy_plots/sfr.pdf"
fig.savefig(sfr_name)
plt.show()

In [ ]:
# early_merger_dict = {sim: {} for sim in sim_lst}

# fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
# plt.subplots_adjust(hspace=0.05, wspace=0.05)


# for i, ax in enumerate(axs.flatten()):
#     if i > 4:
#         ax.axis("off")
#         continue

#     sim = sim_lst[i]

#     tlbt = np.array(gal_dict[sim]["specific_accretion"]["lbt"])
#     t = timez0 - tlbt
#     y = gal_dict[sim]["specific_accretion"]["dat"]
#     y_smooth = gaussian_filter1d(y, sigma=2)

#     ax.plot(tlbt, y, c=colors[i], lw=0.75, zorder=10)
#     ax.axhline(p3, c="r", ls="--", lw=0.5)

#     # find crossings ##########################################################
#     y_shift = y_smooth - p3
#     cross_idx = np.where(np.diff(np.sign(y_shift)) != 0)[0]

#     t_cross = []
#     for idx in cross_idx:
#         t1, t2 = t[idx], t[idx + 1]
#         y1, y2 = y_shift[idx], y_shift[idx + 1]
#         if (y2 - y1) != 0:
#             tc = t1 - y1 * (t2 - t1) / (y2 - y1)
#             t_cross.append(tc)
#     t_cross = np.array(t_cross)

#     # detect early phase #######################################################

#     t_last = None
#     early_intervals = []
#     if len(t_cross) > 0:
#         t_cross = np.sort(t_cross)

#         # find gaps between crossings
#         gaps = np.diff(t_cross)
#         if len(gaps) > 0:
#             # adaptive threshold: identify separation between dense & sparse crossings
#             # gap_threshold = 3 * np.median(gaps)
#             gap_threshold = 2

#             split_idx = np.where(gaps > gap_threshold)[0]

#             if len(split_idx) > 0:
#                 first_cluster = t_cross[: split_idx[0] + 1]
#             else:
#                 first_cluster = t_cross
#         else:
#             first_cluster = t_cross
#         t_last = first_cluster.max()

#     # merger intervals (y > threshold) #######################################################
#     mask = y_smooth > p3
#     edges = np.diff(mask.astype(int))

#     start_idx = np.where(edges == 1)[0] + 1
#     end_idx = np.where(edges == -1)[0] + 1

#     # boundary handling
#     if mask[0]:
#         start_idx = np.insert(start_idx, 0, 0)
#     if mask[-1]:
#         end_idx = np.append(end_idx, len(mask) - 1)

#     intervals = [(t[s], t[e]) for s, e in zip(start_idx, end_idx)]

#     # keep only intervals belonging to early phase
#     if t_last is not None:
#         early_intervals = [(t0, min(t1, t_last)) for t0, t1 in intervals if t0 < t_last]

#     # --- plot early phase ---
#     for t0, t1 in early_intervals:
#         ax.axvspan(timez0 - t0, timez0 - t1, color=colors[i], alpha=0.15)

#     if t_last is not None:
#         ax.axvline(timez0 - t_last, color="k", ls=":", lw=1)
#     early_merger_dict[sim] = np.round(timez0 - t_last, 3)

#     # formatting #############################################################################
#     ax.set_xlim([timez0 - 1, 0])
#     ax.set_ylim(-2, 15)
#     ax.set_xticks([0, 2, 4, 6, 8, 10, 12])

#     ax.text(
#         0.80,
#         0.90,
#         sim,
#         transform=ax.transAxes,
#         fontsize=10,
#         bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.3"),
#     )

#     if i in [0, 3]:
#         ax.set_ylabel("Specific Mass Accretion (Gyr$^{-1}$)")
#     else:
#         ax.set_yticklabels([])

#     if i < 2:
#         ax.set_xticklabels([])

#     if i in [2, 3, 4]:
#         ax.set_xlabel("Lookback Time (Gyr)")


# sa_name = "../data/figures/galaxy_plots/specific_accretion.pdf"
# # fig.tight_layout()
# fig.savefig(sa_name)
# plt.show()


# em_path = "../data/sim_data/early_merger_lbt.json"
# with open(em_path, "w") as f:
#     json.dump(early_merger_dict, f, indent=4)

In [ ]:
early_merger_dict = {sim: {} for sim in sim_lst}

# fig, axs = plt.subplots(3, 2, figsize=(3.35, 4.0))
fig, axs = plt.subplots(3, 2, figsize=(3.5, 3.75))
plt.subplots_adjust(hspace=0.075, wspace=0.075)


for i, ax in enumerate(axs.flatten()):
    if i > 4:
        ax.axis("off")
        continue

    sim = sim_lst[i]

    tlbt = np.array(gal_dict[sim]["specific_accretion"]["lbt"])
    t = timez0 - tlbt
    y = gal_dict[sim]["specific_accretion"]["dat"]
    y_smooth = gaussian_filter1d(y, sigma=2)

    ax.plot(tlbt, y, c=colors[i], lw=0.5, zorder=10)
    ax.axhline(p3, c="r", ls="--", lw=0.5)

    # find crossings ##########################################################
    y_shift = y_smooth - p3
    cross_idx = np.where(np.diff(np.sign(y_shift)) != 0)[0]

    t_cross = []
    for idx in cross_idx:
        t1, t2 = t[idx], t[idx + 1]
        y1, y2 = y_shift[idx], y_shift[idx + 1]
        if (y2 - y1) != 0:
            tc = t1 - y1 * (t2 - t1) / (y2 - y1)
            t_cross.append(tc)
    t_cross = np.array(t_cross)

    # detect early phase #######################################################

    t_last = None
    early_intervals = []
    if len(t_cross) > 0:
        t_cross = np.sort(t_cross)

        # find gaps between crossings
        gaps = np.diff(t_cross)
        if len(gaps) > 0:
            # adaptive threshold: identify separation between dense & sparse crossings
            # gap_threshold = 3 * np.median(gaps)
            gap_threshold = 2

            split_idx = np.where(gaps > gap_threshold)[0]

            if len(split_idx) > 0:
                first_cluster = t_cross[: split_idx[0] + 1]
            else:
                first_cluster = t_cross
        else:
            first_cluster = t_cross
        t_last = first_cluster.max()

    # merger intervals (y > threshold) #######################################################
    mask = y_smooth > p3
    edges = np.diff(mask.astype(int))

    start_idx = np.where(edges == 1)[0] + 1
    end_idx = np.where(edges == -1)[0] + 1

    # boundary handling
    if mask[0]:
        start_idx = np.insert(start_idx, 0, 0)
    if mask[-1]:
        end_idx = np.append(end_idx, len(mask) - 1)

    intervals = [(t[s], t[e]) for s, e in zip(start_idx, end_idx)]

    # keep only intervals belonging to early phase
    if t_last is not None:
        early_intervals = [(t0, min(t1, t_last)) for t0, t1 in intervals if t0 < t_last]

    # --- plot early phase ---
    for t0, t1 in early_intervals:
        ax.axvspan(timez0 - t0, timez0 - t1, color=colors[i], alpha=0.15)

    if t_last is not None:
        ax.axvline(timez0 - t_last, color="k", ls=":", lw=1)
    early_merger_dict[sim] = np.round(timez0 - t_last, 3)

    # formatting #############################################################################
    ax.set_xlim([timez0 - 1, 0])
    ax.set_ylim(-2, 15)
    ax.set_xticks([0, 2, 4, 6, 8, 10, 12])

    ax.text(
        0.75,
        0.80,
        sim,
        transform=ax.transAxes,
        bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.3"),
    )

    print(np.round(gal_dict[sim]["specific_accretion"]["lbt"][0] - (timez0 - t_last), 1))

    if i in [0, 2, 4]:
        # ax.set_ylabel("Specific Mass Accretion (Gyr$^{-1}$)")
        ax.set_ylabel(r"$\dot{{\rm M}}_{\rm h}$ / ${\rm M}_{\rm h}$ (Gyr$^{-1}$)")
    else:
        ax.set_yticklabels([])

    if i < 3:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Lookback Time (Gyr)")


sa_name = "../data/figures/galaxy_plots/specific_accretion.pdf"
# fig.tight_layout()
fig.savefig(sa_name, bbox_inches="tight")
plt.show()


em_path = "../data/sim_data/early_merger_lbt.json"
with open(em_path, "w") as f:
    json.dump(early_merger_dict, f, indent=4)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(6, 12), sharex=True, constrained_layout=True)

# ==========================================================
# Stellar Mass
# ==========================================================
ax = axs[0]

for sim in sim_lst:
    tlbt = np.array(gal_dict[sim]["stellar_mass"]["lbt"])
    sm = gal_dict[sim]["stellar_mass"]["dat"]
    sm_smooth = gaussian_filter1d(sm, sigma=8)

    ax.plot(tlbt, sm_smooth, c=sim_colors[sim], label=sim)

    minidx = np.argmin(np.abs(tlbt - to_dict["cnt"][sim]))
    xmax = to_dict["cnt"][sim]
    ymax = sm_smooth[minidx]

    ax.plot([xmax, xmax], [0, ymax], c=sim_colors[sim], lw=1, ls="--")
    ax.plot([timez0, xmax], [ymax, ymax], c=sim_colors[sim], lw=1, ls="--")

ax.set_yscale("log")
ax.set_ylabel(r"Stellar Mass [M$_\odot$]")
ax.legend(loc="lower right")


# ==========================================================
# Stellar Density
# ==========================================================
ax = axs[1]

for sim in sim_lst:
    tlbt = np.array(gal_dict[sim]["stellar_density"]["lbt"])
    sd = np.array(gal_dict[sim]["stellar_density"]["dat"])

    ax.plot(tlbt, sd, c=sim_colors[sim])

    x = to_dict["cnt"][sim]

    idx = np.argsort(tlbt)
    y = 10 ** np.interp(x, tlbt[idx], np.log10(sd[idx]))

    ax.vlines(
        x,
        ymin=1e3,
        ymax=y,
        colors=sim_colors[sim],
        linestyles="--",
        linewidth=1,
    )
    ax.plot([timez0, x], [y, y], c=sim_colors[sim], lw=1, ls="--")

ax.set_yscale("log")
ax.set_ylim(1e4, 8e8)
ax.set_ylabel(r"$\rho_{\star}$ (R < 3 kpc) [M$_\odot$ kpc$^{-3}$]")


# ==========================================================
# Mass Ratio
# ==========================================================
ax = axs[2]

for sim in sim_lst:
    tlbt = np.array(gal_dict[sim]["mass_ratio"]["lbt"])
    mr = np.array(gal_dict[sim]["mass_ratio"]["dat"])

    ax.plot(tlbt, mr, c=sim_colors[sim])

    x = to_dict["cnt"][sim]

    idx = np.argsort(tlbt)
    y = np.interp(x, tlbt[idx], mr[idx])

    ax.vlines(
        x,
        ymin=0,
        ymax=y,
        colors=sim_colors[sim],
        linestyles="--",
        linewidth=1,
    )
    ax.plot([timez0, x], [y, y], c=sim_colors[sim], lw=1, ls="--")

ax.set_ylim(0, 4.5)
ax.set_ylabel(r"Stellar Mass (R < 3 kpc) / Dark Matter Mass (R < 3 kpc)")
ax.set_xlabel("Lookback Time [Gyr]")


# ==========================================================
# Shared formatting
# ==========================================================
for ax in axs:
    ax.set_xlim(timez0 - 1, 0)

save_name = "../data/figures/galaxy_plots/turn_off_relations.pdf"
fig.savefig(save_name, bbox_inches="tight")

plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(3.5, 2))
fig, ax = plt.subplots(figsize=(8, 6))

for sim in sim_lst:
    # tlbt = np.array(gal_dict[sim]["gas_density"]["lbt"])
    # gd = np.array(gal_dict[sim]["gas_density"]["dat"])

    tlbt = np.array(gal_dict[sim]["bary_ratio"]["lbt"])
    gd = np.array(gal_dict[sim]["bary_ratio"]["dat"])

    ax.plot(tlbt, gd, c=sim_colors[sim])

    minidx = np.argmin(np.abs(tlbt - bf_dict["in_situ"][sim]))
    xmax = bf_dict["in_situ"][sim]
    ymax = gd[minidx]

    ax.plot([xmax, xmax], [0, ymax], c=sim_colors[sim], lw=1, ls="--")
    ax.plot([timez0, xmax], [ymax, ymax], c=sim_colors[sim], lw=1, ls="--")

# ax.set_yscale("log")

ax.set_xlim(13, 0)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(6, 12), sharex=True, constrained_layout=True)

# ==========================================================
# Stellar Mass
# ==========================================================
ax = axs[0]

for sim in sim_lst:
    tlbt = np.array(gal_dict[sim]["stellar_mass"]["lbt"])
    sm = gal_dict[sim]["stellar_mass"]["dat"]
    sm_smooth = gaussian_filter1d(sm, sigma=8)

    ax.plot(tlbt, sm_smooth, c=sim_colors[sim], label=sim)

    minidx = np.argmin(np.abs(tlbt - early_merger_dict[sim]))
    xmax = to_dict["cnt"][sim]
    ymax = sm_smooth[minidx]

    ax.plot([xmax, xmax], [0, ymax], c=sim_colors[sim], lw=1, ls="--")
    ax.plot([timez0, xmax], [ymax, ymax], c=sim_colors[sim], lw=1, ls="--")

ax.set_yscale("log")
ax.set_ylabel(r"Stellar Mass [M$_\odot$]")
ax.legend(loc="lower right")


# ==========================================================
# Stellar Density
# ==========================================================
ax = axs[1]

for sim in sim_lst:
    tlbt = np.array(gal_dict[sim]["stellar_density"]["lbt"])
    sd = np.array(gal_dict[sim]["stellar_density"]["dat"])

    ax.plot(tlbt, sd, c=sim_colors[sim])

    x = early_merger_dict[sim]

    idx = np.argsort(tlbt)
    y = 10 ** np.interp(x, tlbt[idx], np.log10(sd[idx]))

    ax.vlines(
        x,
        ymin=1e3,
        ymax=y,
        colors=sim_colors[sim],
        linestyles="--",
        linewidth=1,
    )
    ax.plot([timez0, x], [y, y], c=sim_colors[sim], lw=1, ls="--")

ax.set_yscale("log")
ax.set_ylim(1e4, 8e8)
ax.set_ylabel(r"$\rho_{\star}$ (R < 3 kpc) [M$_\odot$ kpc$^{-3}$]")


# ==========================================================
# Mass Ratio
# ==========================================================
ax = axs[2]

for sim in sim_lst:
    tlbt = np.array(gal_dict[sim]["mass_ratio"]["lbt"])
    mr = np.array(gal_dict[sim]["mass_ratio"]["dat"])

    ax.plot(tlbt, mr, c=sim_colors[sim])

    early_merger_dict[sim]

    idx = np.argsort(tlbt)
    y = np.interp(x, tlbt[idx], mr[idx])

    ax.vlines(
        x,
        ymin=0,
        ymax=y,
        colors=sim_colors[sim],
        linestyles="--",
        linewidth=1,
    )
    ax.plot([timez0, x], [y, y], c=sim_colors[sim], lw=1, ls="--")

ax.set_ylim(0, 4.5)
ax.set_ylabel(r"Stellar Mass (R < 3 kpc) / Dark Matter Mass (R < 3 kpc)")
ax.set_xlabel("Lookback Time [Gyr]")


# ==========================================================
# Shared formatting
# ==========================================================
for ax in axs:
    ax.set_xlim(timez0 - 1, 0)

save_name = "../data/figures/galaxy_plots/turn_off_relations.pdf"
fig.savefig(save_name, bbox_inches="tight")

plt.show()

In [ ]:
# fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))
# plt.subplots_adjust(hspace=0.05, wspace=0.05)

# index_map = {0: su_dict, 1: early_merger_dict, 2: to_dict["cnt"], 3: bf_dict["in_situ"]}
# label_map = {
#     0: "Spin-Up Lookback Time (Gyr)",
#     1: "End of Early Merger Phase \n Lookback Time (Gyr)",
#     2: "Turn-Off Lookback Time (Gyr)",
#     3: "Survival Boundary Formation \n Lookback Time (Gyr)",
# }

# for j in range(3):
#     for i in range(j, 3):
#         for sim in sim_lst:
#             axs[i, j].scatter(index_map[j][sim], index_map[i + 1][sim], color=sim_colors[sim], label=sim)
#             axs[i, j].plot([0, 14], [0, 14], ls="-", lw=0.25, c="k")

#             axs[i, j].set_xlim(3.5, 12.5)
#             axs[i, j].set_ylim(3.5, 12.5)

#             axs[i, j].set_xticks([6, 8, 10, 12])
#             axs[i, j].set_yticks([6, 8, 10, 12])

#             if i < 2:
#                 axs[i, j].set_xlabel("")
#                 axs[i, j].set_xticklabels([])
#             else:
#                 axs[i, j].set_xlabel(label_map[j])

#             if j > 0:
#                 axs[i, j].set_ylabel("")
#                 axs[i, j].set_yticklabels([])
#             else:
#                 axs[i, j].set_ylabel(label_map[i + 1])

#             if (i == 0) & (j == 0):
#                 axs[i, j].legend(
#                     loc="center right",
#                     frameon=False,
#                     bbox_to_anchor=(1.5, 0.25),
#                 )

# axs[0, 1].axis("off")
# axs[0, 2].axis("off")
# axs[1, 2].axis("off")

# save_name = "../data/figures/galaxy_plots/time_compare.pdf"
# fig.savefig(save_name, bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(nrows=4, ncols=4, figsize=(10, 10))
plt.subplots_adjust(hspace=0.05, wspace=0.05)

index_map = {
    0: su_dict,
    1: early_merger_dict,
    2: to_dict["cnt"],
    3: bf_dict["in_situ"],
    4: bo_dict["in_situ"],
}
label_map = {
    0: "Spin-Up Lookback Time (Gyr)",
    1: "End of Early Merger Phase \n Lookback Time (Gyr)",
    2: "Turn-Off Lookback Time (Gyr)",
    3: "Survival Boundary Formation \n Lookback Time (Gyr)",
    4: "Survival Boundary Onset \n Lookback Time (Gyr)",
}

for j in range(4):
    for i in range(j, 4):
        for sim in sim_lst:
            axs[i, j].scatter(index_map[j][sim], index_map[i + 1][sim], color=sim_colors[sim], label=sim)
            axs[i, j].plot([0, 14], [0, 14], ls="-", lw=0.25, c="k")

            axs[i, j].set_xlim(3.5, 12.5)
            axs[i, j].set_ylim(3.5, 12.5)

            axs[i, j].set_xticks([6, 8, 10, 12])
            axs[i, j].set_yticks([6, 8, 10, 12])

            if i < 3:
                axs[i, j].set_xlabel("")
                axs[i, j].set_xticklabels([])
            else:
                axs[i, j].set_xlabel(label_map[j])

            if j > 0:
                axs[i, j].set_ylabel("")
                axs[i, j].set_yticklabels([])
            else:
                axs[i, j].set_ylabel(label_map[i + 1])

            if (i == 0) & (j == 0):
                axs[i, j].legend(
                    loc="center right",
                    frameon=False,
                    bbox_to_anchor=(1.5, 0.25),
                )

axs[0, 1].axis("off")
axs[0, 2].axis("off")
axs[0, 3].axis("off")

axs[1, 2].axis("off")
axs[1, 3].axis("off")

axs[2, 3].axis("off")

save_name = "../data/figures/galaxy_plots/time_compare.pdf"
fig.savefig(save_name, bbox_inches="tight")

In [ ]:
# situ_flag = None  # True = in-situ, False = ex-situ, None = both
# inmw = True  # True = only MW GCs, False = all alive GCs
# interp = False  # True = Radially interpolate at snapshots, False = Do not

# n_snaps = 39
# n_rbins = 40
# rmax = 20

# r_bins = np.linspace(0, rmax, n_rbins + 1)
# r_centres = 0.5 * (r_bins[:-1] + r_bins[1:])

# cmap = cmr.cosmic_r
# alpha = 0.75

# # Plotting ########################################################

# fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))
# plt.subplots_adjust(hspace=0.05, wspace=0.05)

# axs = axs.flatten()

# # turn off bottom-right panel
# axs[5].axis("off")

# # simulations go in the remaining five panels
# plot_axes = [axs[0], axs[1], axs[2], axs[3], axs[4]]

# vmin, vmax = 1e2, 1e7
# levels = np.logspace(np.log10(vmin), np.log10(vmax), 10)
# norm = LogNorm(vmin=vmin, vmax=vmax)

# for sim, ax in zip(sim_lst, plot_axes):
#     ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
#     ghost_data = h5py.File(ghost_file, "r")

#     tm_sum_in = np.zeros((n_snaps, n_rbins))
#     tm_count_in = np.zeros((n_snaps, n_rbins))

#     for it_id in ghost_data.keys():
#         snap_group = ghost_data[it_id]["snapshots"]

#         for i, snp_id in enumerate(snap_group.keys()):
#             snp_dat = snap_group[snp_id]

#             grp = snp_dat["grpid"][()]
#             nacc = snp_dat["nacc_flag"][()]

#             if situ_flag is None:
#                 grp_msk = np.ones(len(grp), dtype=bool)
#             elif situ_flag:
#                 grp_msk = grp == 0
#             else:
#                 grp_msk = grp != 0

#             if inmw:
#                 mw_msk = nacc.astype(bool)
#             else:
#                 mw_msk = np.ones(len(grp), dtype=bool)

#             msk = grp_msk & mw_msk

#             teigs = snp_dat["tideig"][msk]
#             rs = snp_dat["host.pos.sph"][:, 0][msk]

#             for j in range(n_rbins):
#                 r_msk = (rs >= r_bins[j]) & (rs < r_bins[j + 1])
#                 if np.any(r_msk):
#                     tm_sum_in[i, j] += np.mean(teigs[r_msk])
#                     tm_count_in[i, j] += 1

#     tm_avg_in = np.full_like(tm_sum_in, np.nan)

#     valid = tm_count_in > 0
#     tm_avg_in[valid] = tm_sum_in[valid] / tm_count_in[valid]

#     x = pub_times
#     z_mid = tm_avg_in.T

#     y = np.concatenate(([0.0], r_centres, [rmax]))
#     z = np.vstack([z_mid[0], z_mid, z_mid[-1]])

#     z_interp = z.copy()

#     if interp:
#         for i in range(z.shape[1]):
#             col = z[:, i]
#             m = np.isfinite(col)

#             if m.sum() >= 2:
#                 f = interp1d(
#                     y[m],
#                     col[m],
#                     kind="linear",
#                     bounds_error=False,
#                     fill_value=(col[m][0], col[m][-1]),
#                 )

#                 z_interp[:, i] = f(y)

#     z_interp = np.clip(z_interp, vmin, None)

#     contour = ax.contourf(
#         timez0 - x,
#         y,
#         z_interp,
#         levels=levels,
#         cmap=cmap,
#         norm=norm,
#         alpha=alpha,
#     )

#     for t in x:
#         ax.axvline(timez0 - t, color="grey", lw=0.5, ls="-")

#     # ax.axvline(su_dict[sim], color="r", lw=1, ls="--")
#     ax.axvline(to_dict["cnt"][sim], c="g", ls="--", lw=2)
#     ax.axvline(bf_dict["in_situ"][sim], c="r", ls=":")
#     ax.axvline(early_merger_dict[sim], c="k", ls="-")
#     ax.axvline(su_dict[sim], c="b", ls="-.")

#     ax.axhline(3, c="k", ls="-", lw=0.5)

#     ax.set_xlim(13, 0)
#     ax.set_ylim(0.1, rmax)

#     ax.text(
#         0.97,
#         0.95,
#         sim,
#         transform=ax.transAxes,
#         ha="right",
#         va="top",
#         color="k",
#         # fontsize=12,
#         # bbox=dict(
#         #     facecolor="white",
#         #     edgecolor="none",
#         #     alpha=0.8,
#         #     pad=2,
#         # ),
#     )

# for i, ax in enumerate(axs.flatten()):
#     if i < 2:
#         ax.set_xticklabels([])
#     else:
#         ax.set_xlabel("Lookback Time (Gyr)")

#     if i not in [0, 3]:
#         ax.set_yticklabels([])
#     else:
#         ax.set_ylabel(r"r (kpc)")


# # Colourbar ################################################################

# pos = axs[5].get_position()

# cb_thickness = 0.10 * pos.width

# cax = fig.add_axes(
#     [
#         pos.x0,  # left
#         pos.y0 + 0.8 * pos.height - 0.5 * cb_thickness,  # centred in y
#         pos.width,  # full width of slot
#         cb_thickness,  # colourbar height
#     ]
# )

# cbar = fig.colorbar(
#     contour,
#     cax=cax,
#     orientation="horizontal",
# )

# cbar.set_label(r"$\lambda_e\;(\mathrm{Gyr}^{-2})$")
# cbar.locator = LogLocator(base=10.0)
# cbar.update_ticks()

# save_name = "../data/figures/galaxy_plots/tidal_field.pdf"
# fig.savefig(save_name, bbox_inches="tight")

In [ ]:
situ_flag = None  # True = in-situ, False = ex-situ, None = both
inmw = True  # True = only MW GCs, False = all alive GCs
interp = False  # True = Radially interpolate at snapshots, False = Do not

n_snaps = 39
n_rbins = 40
rmax = 20

r_bins = np.linspace(0, rmax, n_rbins + 1)
r_centres = 0.5 * (r_bins[:-1] + r_bins[1:])

cmap = cmr.cosmic_r
alpha = 0.75

# Plotting ########################################################

fig, axs = plt.subplots(3, 2, figsize=(3.5, 3.75))
plt.subplots_adjust(hspace=0.075, wspace=0.075)

axs = axs.flatten()

# turn off bottom-right panel
axs[5].axis("off")

# simulations go in the remaining five panels
plot_axes = [axs[0], axs[1], axs[2], axs[3], axs[4]]

vmin, vmax = 1e2, 1e7
levels = np.logspace(np.log10(vmin), np.log10(vmax), 10)
norm = LogNorm(vmin=vmin, vmax=vmax)

# choose label-box colour index here
cidx = 1

for sim, ax in zip(sim_lst, plot_axes):
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    tm_sum_in = np.zeros((n_snaps, n_rbins))
    tm_count_in = np.zeros((n_snaps, n_rbins))

    for it_id in ghost_data.keys():
        snap_group = ghost_data[it_id]["snapshots"]

        for i, snp_id in enumerate(snap_group.keys()):
            snp_dat = snap_group[snp_id]

            grp = snp_dat["grpid"][()]
            nacc = snp_dat["nacc_flag"][()]

            if situ_flag is None:
                grp_msk = np.ones(len(grp), dtype=bool)
            elif situ_flag:
                grp_msk = grp == 0
            else:
                grp_msk = grp != 0

            if inmw:
                mw_msk = nacc.astype(bool)
            else:
                mw_msk = np.ones(len(grp), dtype=bool)

            msk = grp_msk & mw_msk

            teigs = snp_dat["tideig"][msk]
            rs = snp_dat["host.pos.sph"][:, 0][msk]

            for j in range(n_rbins):
                r_msk = (rs >= r_bins[j]) & (rs < r_bins[j + 1])

                if np.any(r_msk):
                    tm_sum_in[i, j] += np.mean(teigs[r_msk])
                    tm_count_in[i, j] += 1

    tm_avg_in = np.full_like(tm_sum_in, np.nan)

    valid = tm_count_in > 0
    tm_avg_in[valid] = tm_sum_in[valid] / tm_count_in[valid]

    x = pub_times
    z_mid = tm_avg_in.T

    y = np.concatenate(([0.0], r_centres, [rmax]))
    z = np.vstack([z_mid[0], z_mid, z_mid[-1]])

    z_interp = z.copy()

    if interp:
        for i in range(z.shape[1]):
            col = z[:, i]
            m = np.isfinite(col)

            if m.sum() >= 2:
                f = interp1d(
                    y[m],
                    col[m],
                    kind="linear",
                    bounds_error=False,
                    fill_value=(col[m][0], col[m][-1]),
                )

                z_interp[:, i] = f(y)

    z_interp = np.clip(z_interp, vmin, None)

    contour = ax.contourf(
        timez0 - x,
        y,
        z_interp,
        levels=levels,
        cmap=cmap,
        norm=norm,
        alpha=alpha,
    )

    for t in x:
        ax.axvline(timez0 - t, color="grey", lw=0.5, ls="-")

    ax.axvline(to_dict["cnt"][sim], c="g", ls="--", lw=0.75)
    ax.axvline(su_dict[sim], c="b", ls="-.", lw=0.75)
    ax.axvline(bf_dict["in_situ"][sim], c="r", ls=":", lw=0.75)
    ax.axvline(bo_dict["in_situ"][sim], c="r", ls=":", lw=0.75)
    ax.axvline(early_merger_dict[sim], c="k", ls="-", lw=0.75)

    ax.axhline(3, c="k", ls="-", lw=0.75)

    ax.set_xlim(13, 0)
    ax.set_ylim(0.1, rmax)

    # Actual contour-bin colours
    mids = np.sqrt(levels[:-1] * levels[1:])
    bin_colours = contour.cmap(contour.norm(mids))

    cidx = 2

    rgb = np.array(bin_colours[cidx][:3])
    rgb = alpha * rgb + (1 - alpha) * np.ones(3)

    ax.text(
        0.75,
        0.80,
        sim,
        transform=ax.transAxes,
        bbox=dict(
            facecolor=rgb,
            edgecolor="none",
            boxstyle="round,pad=0.3",
        ),
    )

for i, ax in enumerate(axs.flatten()):
    ax.set_xticks([0, 2, 4, 6, 8, 10, 12])

    if i < 3:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Lookback Time (Gyr)")

    if i not in [0, 2, 4]:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel(r"r (kpc)")
# Colourbar ################################################################

pos = axs[5].get_position()

cb_thickness = 0.075 * pos.width

cax = fig.add_axes(
    [
        pos.x0,
        pos.y0 + 0.5 * pos.height - 0.5 * cb_thickness,
        pos.width,
        cb_thickness,
    ]
)

cbar = fig.colorbar(
    contour,
    cax=cax,
    orientation="horizontal",
)

cbar.set_label(r"$\lambda_e\;(\mathrm{Gyr}^{-2})$")
cbar.locator = LogLocator(base=10.0)
cbar.update_ticks()

save_name = "../data/figures/galaxy_plots/tidal_field.pdf"
fig.savefig(save_name, bbox_inches="tight")
plt.show()


In [ ]:
# Average tidal field strength in the region:
# r < 3 kpc and lookback time > early_merger_dict[sim]

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    tm_sum_region = 0.0
    tm_count_region = 0

    for it_id in ghost_data.keys():
        snap_group = ghost_data[it_id]["snapshots"]

        for i, snp_id in enumerate(snap_group.keys()):
            snp_dat = snap_group[snp_id]

            grp = snp_dat["grpid"][()]
            nacc = snp_dat["nacc_flag"][()]

            if situ_flag is None:
                grp_msk = np.ones(len(grp), dtype=bool)
            elif situ_flag:
                grp_msk = grp == 0
            else:
                grp_msk = grp != 0

            if inmw:
                mw_msk = nacc.astype(bool)
            else:
                mw_msk = np.ones(len(grp), dtype=bool)

            msk = grp_msk & mw_msk

            teigs = snp_dat["tideig"][msk]
            rs = snp_dat["host.pos.sph"][:, 0][msk]

            lookback = timez0 - pub_times[i]

            region_mask = (rs < 3.0) & (lookback > early_merger_dict[sim])

            tm_sum_region += np.sum(teigs[region_mask])
            tm_count_region += np.count_nonzero(region_mask)

    avg_tidal = tm_sum_region / tm_count_region

    # print(f"{sim:10s}  <lambda_e> = {avg_tidal:.1e} Gyr^-2 (N={tm_count_region:,})")
    print(f"{sim:10s}  <lambda_e> = {avg_tidal:.1e} = 10^ {np.round(np.log10(avg_tidal), 2)} Gyr^-2")
    # print(sim, np.round(np.log10(avg_tidal), 2))